<a href="https://colab.research.google.com/github/AbdullahRasheed452/ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
import os, subprocess

REPO_URL = "https://github.com/AbdullahRasheed452/ML-Internship"
REPO_DIR = "ML-Internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
if os.getcwd().split("/")[-1] != REPO_DIR:
    os.chdir(REPO_DIR)

In [16]:
%pip -q install duckdb huggingface_hub

In [17]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('HF_TOKEN')

In [18]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [19]:
# Signal 1: does CTR go down as position gets worse?
# This is the idea behind FlyRank's CTR fix flag
# We group pages into position buckets and check the average CTR in each

In [20]:
signal1_check = con.sql(f"""
    SELECT
        CASE
            WHEN f.gsc_avg_position <= 3 THEN '1_top3'
            WHEN f.gsc_avg_position <= 10 THEN '2_top10'
            WHEN f.gsc_avg_position <= 20 THEN '3_top20'
            ELSE '4_below20'
        END AS position_bucket,
        COUNT(*) AS n,
        AVG(CASE WHEN f.gsc_impressions > 0 THEN f.gsc_clicks * 1.0 / f.gsc_impressions ELSE NULL END) AS avg_ctr
    FROM {TABLES['fact_daily']} f
    WHERE f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
      AND f.gsc_impressions > 0
    GROUP BY position_bucket
    ORDER BY position_bucket
""").df()

signal1_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,avg_ctr
0,1_top3,727362,0.004756
1,2_top10,1456122,0.003473
2,3_top20,519223,0.002770
3,4_below20,908354,0.001289


In [21]:
# Result: average CTR drops as position gets worse
# top3 = 0.0048, top10 = 0.0035, top20 = 0.0028, below20 = 0.0013
# Verdict: CONFIRMED
# CTR really does fall as position gets worse, this signal is safe to use

In [22]:
# Signal 2: are stale pages still getting real visibility?
# This is the idea behind FlyRank's refresh flag
# We check if pages not updated in a long time still have meaningful impressions

In [23]:
signal2_check = con.sql(f"""
    SELECT
        CASE
            WHEN DATE_DIFF('day', c.last_optimized_date, DATE '2026-03-31') >= 180 THEN '1_stale_180plus'
            WHEN DATE_DIFF('day', c.last_optimized_date, DATE '2026-03-31') >= 90 THEN '2_aging_90to180'
            ELSE '3_fresh_under90'
        END AS freshness_bucket,
        COUNT(*) AS n,
        AVG(f.gsc_impressions) AS avg_impressions
    FROM {TABLES['fact_daily']} f
    JOIN {TABLES['dim_content']} c ON f.content_hash_id = c.content_hash_id
    WHERE f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
      AND c.last_optimized_date IS NOT NULL
    GROUP BY freshness_bucket
    ORDER BY freshness_bucket
""").df()

signal2_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,freshness_bucket,n,avg_impressions
0,3_fresh_under90,1286535,146.303673


In [24]:
# Result: only fresh pages show up, stale groups are empty
# Verdict: FALSE
# staleness has no real variation in this slice, dropped from the rule

In [25]:
# My rule: score pages higher when their CTR is low relative to their position
# tier, and they have enough impressions to matter
# Reason code: low_ctr_for_position
# Action: review title and meta description

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [26]:
# Score idea:
# Compare each page's CTR to the normal CTR for its position
# A bigger gap means a bigger opportunity, so it gets a higher score
# Only pages with at least 50 impressions are scored, to skip noise

In [27]:
scored = con.sql(f"""
    WITH tier_avg AS (
        SELECT
            CASE
                WHEN gsc_avg_position <= 3 THEN '1_top3'
                WHEN gsc_avg_position <= 10 THEN '2_top10'
                WHEN gsc_avg_position <= 20 THEN '3_top20'
                ELSE '4_below20'
            END AS position_bucket,
            AVG(CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions ELSE NULL END) AS tier_avg_ctr
        FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
          AND gsc_impressions > 0
        GROUP BY position_bucket
    ),
    page_stats AS (
        SELECT
            content_hash_id,
            AVG(gsc_impressions) AS avg_impressions,
            AVG(gsc_avg_position) AS avg_position,
            AVG(CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions ELSE NULL END) AS page_ctr,
            CASE
                WHEN AVG(gsc_avg_position) <= 3 THEN '1_top3'
                WHEN AVG(gsc_avg_position) <= 10 THEN '2_top10'
                WHEN AVG(gsc_avg_position) <= 20 THEN '3_top20'
                ELSE '4_below20'
            END AS position_bucket
        FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
        GROUP BY content_hash_id
        HAVING AVG(gsc_impressions) >= 50
    )
    SELECT
        p.content_hash_id,
        p.avg_impressions,
        p.avg_position,
        p.page_ctr,
        t.tier_avg_ctr,
        (t.tier_avg_ctr - p.page_ctr) AS ctr_gap,
        'low_ctr_for_position' AS reason_code,
        'review title and meta description' AS action
    FROM page_stats p
    JOIN tier_avg t ON p.position_bucket = t.position_bucket
    WHERE p.page_ctr IS NOT NULL
    ORDER BY ctr_gap DESC
""").df()

scored.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,avg_impressions,avg_position,page_ctr,tier_avg_ctr,ctr_gap,reason_code,action
0,content_1d3cfc8b0ead76cc,51.153846,1.326454,0.0,0.004756,0.004756,low_ctr_for_position,review title and meta description
1,content_449248826d470a5c,53.750000,2.390440,0.0,0.004756,0.004756,low_ctr_for_position,review title and meta description
2,content_cece1be7218442fe,84.000000,2.975066,0.0,0.004756,0.004756,low_ctr_for_position,review title and meta description
3,content_853fe3ebbe3d1a36,56.419355,2.520966,0.0,0.004756,0.004756,low_ctr_for_position,review title and meta description
4,content_90b26f2ec5592253,69.838710,2.886568,0.0,0.004756,0.004756,low_ctr_for_position,review title and meta description
5,content_e1c68d806307790b,90.000000,2.097258,0.0,0.004756,0.004756,low_ctr_for_position,review title and meta description
6,content_94f97aff825cdf48,62.161290,2.585849,0.0,0.004756,0.004756,low_ctr_for_position,review title and meta description
7,content_886f67aee6fbac79,55.903226,2.580669,0.0,0.004756,0.004756,low_ctr_for_position,review title and meta description
8,content_d068ff72670d5e55,54.870968,1.842195,0.0,0.004756,0.004756,low_ctr_for_position,review title and meta description
9,content_66cb7a06190f8b36,64.000000,2.609933,0.0,0.004756,0.004756,low_ctr_for_position,review title and meta description


In [28]:
# Save the ranked queue
import os
os.makedirs("work/outputs", exist_ok=True)
scored.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Saved {len(scored)} rows to work/outputs/baseline_action_score.csv")

Saved 35674 rows to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [29]:
top20 = scored.sort_values(['ctr_gap', 'avg_impressions'], ascending=[False, False]).head(20)
top20

,content_hash_id,avg_impressions,avg_position,page_ctr,tier_avg_ctr,ctr_gap,reason_code,action
84,content_fa17add7836d36c3,406.064516,1.902457,0.0,0.004756,0.004756,low_ctr_for_position,review title and meta description
140,content_d397987113cb84a0,318.935484,1.953104,0.0,0.004756,0.004756,low_ctr_for_position,review title and meta description
132,content_a27b382f00aa75c6,249.548387,2.259812,0.0,0.004756,0.004756,low_ctr_for_position,review title and meta description
137,content_83167156f76e33e5,220.225806,1.136714,0.0,0.004756,0.004756,low_ctr_for_position,review title and meta description
134,content_1bc8782404e3b132,186.838710,2.264276,0.0,0.004756,0.004756,low_ctr_for_position,review title and meta description
133,content_76a6fa55e21323b3,162.612903,2.855291,0.0,0.004756,0.004756,low_ctr_for_position,review title and meta description
32,content_9cec93fc44a7ab41,152.967742,1.852298,0.0,0.004756,0.004756,low_ctr_for_position,review title and meta description
97,content_fd1d19e381fc653e,139.322581,2.789944,0.0,0.004756,0.004756,low_ctr_for_position,review title and meta description
146,content_cc24743a90439e14,136.290323,1.913571,0.0,0.004756,0.004756,low_ctr_for_position,review title and meta description
23,content_db01d94616cdb80d,135.548387,0.911944,0.0,0.004756,0.004756,low_ctr_for_position,review title and meta description


In [30]:
# Top 20 review
# All 20 rows share the same pattern: real impressions (50+), zero recorded
# clicks, at a position where some clicks would normally be expected
# Action: review title and meta description for each of these pages
# Why it's here: page has enough visibility to matter but is not converting
# any of that visibility into clicks at all
# What would make it wrong: if the zero click count is a tracking or data
# gap rather than a real user behavior, or if the query intent for that
# page does not match a clickable result type (like an image heavy page)

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [31]:
# Weak picks
# Every row in the top 20 is tied at the same ctr_gap value, since they
# all have zero clicks, so the ranking within this group is not very
# meaningful, a page with 50 impressions and 0 clicks looks the same as
# a page with 250 impressions and 0 clicks
# A stronger version of this rule would break ties using impressions or
# add a minimum click threshold to separate real opportunities from noise

# Leakage check
# No product flags like health_score or priority_score were used
# No future window data was used, only March 2026 data, the same month
# used to build the score
# The rule only uses features known at the time, like impressions,
# position, and CTR

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.